# final notebook

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import time
import warnings
import soccerdata as sd
from datetime import datetime
import re
import difflib
import unicodedata

## read each season data

In [ ]:
data_merged_gw_2016_17 = pd.read_csv('data/2016-17/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2017_18 = pd.read_csv('data/2017-18/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2018_19 = pd.read_csv('data/2018-19/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2019_20 = pd.read_csv('data/2019-20/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2020_21 = pd.read_csv('data/2020-21/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2021_22 = pd.read_csv('data/2021-22/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2022_23 = pd.read_csv('data/2022-23/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2023_24 = pd.read_csv('data/2023-24/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2024_25 = pd.read_csv('data/2024-25/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2025_26 = pd.read_csv('data/2025-26/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')

### note here that currently I am only making the actions not the analysis I did before to know what to do

## adding position column 

In [ ]:
# Add position column to merged gameweek data based on element_type from player raw data
def add_position_to_merged_gw(merged_gw_df, cleaned_players_df):
    # Map element_type to position names
    element_type_to_position = {
        1: 'Goalkeeper',
        2: 'Defender',
        3: 'Midfielder',
        4: 'Forward'
    }
    # create a new column 'position' in cleaned players dataframe
    cleaned_players_df['position'] = cleaned_players_df['element_type'].map(element_type_to_position)
    # create a mapping from player id to position
    player_id_to_position = dict(zip(cleaned_players_df['id'], cleaned_players_df['position']))
    # add the position column to the merged gw dataframe
    merged_gw_df['position'] = merged_gw_df['element'].map(player_id_to_position)
    return merged_gw_df


### load the players tables for the target seasons

In [ ]:
# first load the raw player data for each season until 2019-20
data_players_2016_17 = pd.read_csv('data/2016-17/players_raw.csv', encoding='latin-1', on_bad_lines='skip')
data_players_2017_18 = pd.read_csv('data/2017-18/players_raw.csv', encoding='latin-1', on_bad_lines='skip')
data_players_2018_19 = pd.read_csv('data/2018-19/players_raw.csv', encoding='latin-1', on_bad_lines='skip')
data_players_2019_20 = pd.read_csv('data/2019-20/players_raw.csv', encoding='latin-1', on_bad_lines='skip')

## add the position column

In [ ]:
# add the position column to each season's player data
data_merged_gw_2016_17 = add_position_to_merged_gw(data_merged_gw_2016_17, data_players_2016_17)
data_merged_gw_2017_18 = add_position_to_merged_gw(data_merged_gw_2017_18, data_players_2017_18)
data_merged_gw_2018_19 = add_position_to_merged_gw(data_merged_gw_2018_19, data_players_2018_19)
data_merged_gw_2019_20 = add_position_to_merged_gw(data_merged_gw_2019_20, data_players_2019_20)

## adding team name to the dataset

In [ ]:
# Add team names to seasons 2016-17 through 2019-20 using master team list
# Process:
# 1. Load master team list (contains season → team_id → team_name mapping)
# 2. Map player_id → team_id from player raw data
# 3. Map team_id → team_name from master list
# 4. Add team_name column to merged GW data
data_master_team_list = pd.read_csv('data/master_team_list.csv', encoding='latin-1', on_bad_lines='skip')
def add_team_name_to_merged_gw(merged_gw_df, players_raw_df, master_team_list_df, season):
    # filter the master team list for the given season
    season_team_list = master_team_list_df[master_team_list_df['season'] == season]
    # create a mapping from team id to team name
    team_id_to_name = dict(zip(season_team_list['team'], season_team_list['team_name']))
    # create a mapping from player id to team id
    player_id_to_team_id = dict(zip(players_raw_df['id'], players_raw_df['team']))
    # create a mapping from player id to team name
    player_id_to_team_name = {player_id: team_id_to_name.get(team_id, 'Unknown') for player_id, team_id in player_id_to_team_id.items()}

    # add the team name column to the merged gw dataframe    return merged_gw_df
    merged_gw_df['team'] = merged_gw_df['element'].map(player_id_to_team_name)

In [ ]:
# apply the function to each season's merged gw data
add_team_name_to_merged_gw(data_merged_gw_2016_17, data_players_2016_17, data_master_team_list, '2016-17')
add_team_name_to_merged_gw(data_merged_gw_2017_18, data_players_2017_18, data_master_team_list, '2017-18')
add_team_name_to_merged_gw(data_merged_gw_2018_19, data_players_2018_19, data_master_team_list, '2018-19')
add_team_name_to_merged_gw(data_merged_gw_2019_20, data_players_2019_20, data_master_team_list, '2019-20')

# ensuring all the seasons has the same attributes

In [ ]:
# Drop the xP (expected points) column from seasons 2020-21 through 2025-26 for consistency
data_merged_gw_2020_21 = data_merged_gw_2020_21.drop(columns=['xP'])
data_merged_gw_2021_22 = data_merged_gw_2021_22.drop(columns=['xP'])
data_merged_gw_2022_23 = data_merged_gw_2022_23.drop(columns=['xP'])
data_merged_gw_2023_24 = data_merged_gw_2023_24.drop(columns=['xP'])
data_merged_gw_2024_25 = data_merged_gw_2024_25.drop(columns=['xP'])
data_merged_gw_2025_26 = data_merged_gw_2025_26.drop(columns=['xP'])
# Drop columns not available across all seasons (2016-17 to 2018-19)
data_merged_gw_2016_17 = data_merged_gw_2016_17.drop(columns=['attempted_passes','big_chances_created','big_chances_missed','completed_passes','dribbles','ea_index','errors_leading_to_goal','errors_leading_to_goal_attempt','fouls','key_passes','kickoff_time_formatted','loaned_in','loaned_out','offside','open_play_crosses','penalties_conceded','tackled','target_missed','winning_goals'])
data_merged_gw_2017_18 = data_merged_gw_2017_18.drop(columns=['attempted_passes','big_chances_created','big_chances_missed','completed_passes','dribbles','ea_index','errors_leading_to_goal','errors_leading_to_goal_attempt','fouls','key_passes','kickoff_time_formatted','loaned_in','loaned_out','offside','open_play_crosses','penalties_conceded','tackled','target_missed','winning_goals'])
data_merged_gw_2018_19 = data_merged_gw_2018_19.drop(columns=['attempted_passes','big_chances_created','big_chances_missed','completed_passes','dribbles','ea_index','errors_leading_to_goal','errors_leading_to_goal_attempt','fouls','key_passes','kickoff_time_formatted','loaned_in','loaned_out','offside','open_play_crosses','penalties_conceded','tackled','target_missed','winning_goals'])
# Drop from 2022 the xp and that stuff
data_merged_gw_2022_23 = data_merged_gw_2022_23.drop(columns=['expected_goals','expected_assists','expected_goals_conceded','expected_goal_involvements','starts'])
data_merged_gw_2023_24 = data_merged_gw_2023_24.drop(columns=['expected_goals','expected_assists','expected_goals_conceded','expected_goal_involvements','starts'])
data_merged_gw_2024_25 = data_merged_gw_2024_25.drop(columns=['expected_goals','expected_assists','expected_goals_conceded','expected_goal_involvements','starts'])
data_merged_gw_2025_26 = data_merged_gw_2025_26.drop(columns=['expected_goals','expected_assists','expected_goals_conceded','expected_goal_involvements','starts'])
# Drop modified in the last two seasons
data_merged_gw_2024_25 = data_merged_gw_2024_25.drop(columns=['modified'])
data_merged_gw_2025_26 = data_merged_gw_2025_26.drop(columns=['modified'])
# drop the id from the first two seasons
data_merged_gw_2016_17 = data_merged_gw_2016_17.drop(columns=['id'])
data_merged_gw_2017_18 = data_merged_gw_2017_18.drop(columns=['id'])
data_merged_gw_2018_19 = data_merged_gw_2018_19.drop(columns=['id'])

## adding defensive contribution

In [ ]:
# Calculate defensive_contribution for seasons 2016-17 to 2024-25
# FPL scoring rules:
# - Defenders: clearances_blocks_interceptions + tackles
# - Midfielders/Forwards: clearances_blocks_interceptions + tackles + recoveries
def add_defensive_contribution(merged_gw_df):
    def calculate_defensive_contribution(row):
        position = row['position']
        clearances = row.get('clearances_blocks_interceptions', 0)
        tackles = row.get('tackles', 0)
        recoveries = row.get('recoveries', 0)
        if position == 'Defender':
            return clearances + tackles
        elif position in ['Midfielder', 'Forward']:
            return clearances + tackles + recoveries
        else:
            return 0
    merged_gw_df['defensive_contribution'] = merged_gw_df.apply(calculate_defensive_contribution, axis=1)
    return merged_gw_df

In [ ]:
# Add placeholder columns (value 0) for defensive stats not tracked in seasons 2019-20 to 2024-25
# This ensures consistent schema across all seasons before calculating defensive_contribution
data_merged_gw_2019_20['clearances_blocks_interceptions'] = 0
data_merged_gw_2019_20['recoveries'] = 0
data_merged_gw_2019_20['tackles'] = 0
data_merged_gw_2020_21['clearances_blocks_interceptions'] = 0
data_merged_gw_2020_21['recoveries'] = 0
data_merged_gw_2020_21['tackles'] = 0
data_merged_gw_2021_22['clearances_blocks_interceptions'] = 0
data_merged_gw_2021_22['recoveries'] = 0
data_merged_gw_2021_22['tackles'] = 0
data_merged_gw_2022_23['clearances_blocks_interceptions'] = 0
data_merged_gw_2022_23['recoveries'] = 0
data_merged_gw_2022_23['tackles'] = 0
data_merged_gw_2023_24['clearances_blocks_interceptions'] = 0
data_merged_gw_2023_24['recoveries'] = 0
data_merged_gw_2023_24['tackles'] = 0
data_merged_gw_2024_25['clearances_blocks_interceptions'] = 0
data_merged_gw_2024_25['recoveries'] = 0
data_merged_gw_2024_25['tackles'] = 0
data_merged_gw_2016_17 = add_defensive_contribution(data_merged_gw_2016_17)
data_merged_gw_2017_18 = add_defensive_contribution(data_merged_gw_2017_18)
data_merged_gw_2018_19 = add_defensive_contribution(data_merged_gw_2018_19)
data_merged_gw_2019_20 = add_defensive_contribution(data_merged_gw_2019_20)
data_merged_gw_2020_21 = add_defensive_contribution(data_merged_gw_2020_21)
data_merged_gw_2021_22 = add_defensive_contribution(data_merged_gw_2021_22)
data_merged_gw_2022_23 = add_defensive_contribution(data_merged_gw_2022_23)
data_merged_gw_2023_24 = add_defensive_contribution(data_merged_gw_2023_24)
data_merged_gw_2024_25 = add_defensive_contribution(data_merged_gw_2024_25)

### verify that all the cols are identical now

In [ ]:
# compare the columns of all these datasets
merged_2016_17_columns = set(data_merged_gw_2016_17.columns.tolist())
merged_2017_18_columns = set(data_merged_gw_2017_18.columns.tolist())
merged_2018_19_columns = set(data_merged_gw_2018_19.columns.tolist())
merged_2019_20_columns = set(data_merged_gw_2019_20.columns.tolist())
merged_2020_21_columns = set(data_merged_gw_2020_21.columns.tolist())
merged_2021_22_columns = set(data_merged_gw_2021_22.columns.tolist())
merged_2022_23_columns = set(data_merged_gw_2022_23.columns.tolist())
merged_2023_24_columns = set(data_merged_gw_2023_24.columns.tolist())
merged_2024_25_columns = set(data_merged_gw_2024_25.columns.tolist())
merged_2025_26_columns = set(data_merged_gw_2025_26.columns.tolist())
# find the common columns across all seasons
common_merged_columns_all_seasons = merged_2016_17_columns.intersection(merged_2017_18_columns).intersection(merged_2018_19_columns).intersection(merged_2019_20_columns).intersection(merged_2020_21_columns).intersection(merged_2021_22_columns).intersection(merged_2022_23_columns).intersection(merged_2023_24_columns).intersection(merged_2024_25_columns).intersection(merged_2025_26_columns)
print("Common Columns Across All Seasons:", sorted(common_merged_columns_all_seasons))
# find the unique columns in each season compared to the common columns
unique_2016_17_columns = merged_2016_17_columns - common_merged_columns_all_seasons
unique_2017_18_columns = merged_2017_18_columns - common_merged_columns_all_seasons
unique_2018_19_columns = merged_2018_19_columns - common_merged_columns_all_seasons
unique_2019_20_columns = merged_2019_20_columns - common_merged_columns_all_seasons
unique_2020_21_columns = merged_2020_21_columns - common_merged_columns_all_seasons
unique_2021_22_columns = merged_2021_22_columns - common_merged_columns_all_seasons
unique_2022_23_columns = merged_2022_23_columns - common_merged_columns_all_seasons
unique_2023_24_columns = merged_2023_24_columns - common_merged_columns_all_seasons
unique_2024_25_columns = merged_2024_25_columns - common_merged_columns_all_seasons
unique_2025_26_columns = merged_2025_26_columns - common_merged_columns_all_seasons
print("Unique Columns in 2016-17:", sorted(unique_2016_17_columns))
print("Unique Columns in 2017-18:", sorted(unique_2017_18_columns))
print("Unique Columns in 2018-19:", sorted(unique_2018_19_columns))
print("Unique Columns in 2019-20:", sorted(unique_2019_20_columns))
print("Unique Columns in 2020-21:", sorted(unique_2020_21_columns))
print("Unique Columns in 2021-22:", sorted(unique_2021_22_columns))
print("Unique Columns in 2022-23:", sorted(unique_2022_23_columns))
print("Unique Columns in 2023-24:", sorted(unique_2023_24_columns))
print("Unique Columns in 2024-25:", sorted(unique_2024_25_columns))
print("Unique Columns in 2025-26:", sorted(unique_2025_26_columns))


# hsitorical points adjustment

## here there is a key decision: 
### the points of the previous seasons will be modified to work with the same system as the new rules (adding points for defensive contribution)

### consider moving this to the end (after merging with the defensive data)

1. Standardize position values across all seasons2. Adjust points for seasons 2016-17 to 2018-19 to match current FPL defensive contribution scoring

In [ ]:
# Adjust historical points (2016-17 to 2018-19) to align with current FPL scoring system
# Add 2 bonus points when:
# - Defenders reach defensive_contribution >= 10
# - Midfielders/Forwards reach defensive_contribution >= 12
def modify_points(merged_gw_df):
    def calculate_modified_points(row):
        points = row['total_points']
        position = row['position']
        defensive_contribution = row['defensive_contribution']
        if position == 'Defender' and defensive_contribution >= 10:
            points += 2
        elif position in ['Midfielder', 'Forward'] and defensive_contribution >= 12:
            points += 2
        return points
    merged_gw_df['total_points'] = merged_gw_df.apply(calculate_modified_points, axis=1)
    return merged_gw_df
data_merged_gw_2016_17 = modify_points(data_merged_gw_2016_17)
data_merged_gw_2017_18 = modify_points(data_merged_gw_2017_18)
data_merged_gw_2018_19 = modify_points(data_merged_gw_2018_19)

### note here that you need to run modify points of the seasons from 2019 to 2025 when you merge with the defensive data

# merging all seasons data in a single dataset

In [ ]:
# Add season identifier to each dataset before merging
data_merged_gw_2016_17['season'] = '2016-17'
data_merged_gw_2017_18['season'] = '2017-18'
data_merged_gw_2018_19['season'] = '2018-19'
data_merged_gw_2019_20['season'] = '2019-20'
data_merged_gw_2020_21['season'] = '2020-21'
data_merged_gw_2021_22['season'] = '2021-22'
data_merged_gw_2022_23['season'] = '2022-23'
data_merged_gw_2023_24['season'] = '2023-24'
data_merged_gw_2024_25['season'] = '2024-25'
data_merged_gw_2025_26['season'] = '2025-26'
# concatenating all seasons data into a single dataframe
all_seasons_data = pd.concat([data_merged_gw_2016_17, data_merged_gw_2017_18, data_merged_gw_2018_19, data_merged_gw_2019_20, data_merged_gw_2020_21, data_merged_gw_2021_22, data_merged_gw_2022_23, data_merged_gw_2023_24, data_merged_gw_2024_25, data_merged_gw_2025_26], ignore_index=True)
print("All Seasons Data Sample:")
print(all_seasons_data.sample(10))

## standarizing the position across seasons

In [ ]:
# Standardize position values to short codes for consistency
all_seasons_data['position'] = all_seasons_data['position'].replace({'Goalkeeper': 'GK', 'Defender': 'DEF', 'Midfielder': 'MID', 'Forward': 'FWD'})

## converting the opponent team from id to name

In [ ]:
# Convert opponent_team from team IDs to team names using master team list
# Create season-specific mappings of team_id -> team_name
team_id_name_mapping = {}
for season in all_seasons_data['season'].unique():
    season_team_data = data_master_team_list[data_master_team_list['season'] == season]
    team_id_name_mapping[season] = dict(zip(season_team_data['team'], season_team_data['team_name']))
# replace the opponent_team in all_seasons_data based on the season and the team id
def replace_opponent_team(row):
    season = row['season']
    team_id = row['opponent_team']
    return team_id_name_mapping[season].get(team_id, team_id)
all_seasons_data['opponent_team'] = all_seasons_data.apply(replace_opponent_team, axis=1)

# saving the current state of data as csv

In [ ]:
# save all seasons data to a csv file
all_seasons_data.to_csv('all_seasons_data.csv', index=False, encoding='latin-1')

# here we add the game number feature then we make the merging with defensive stats

## here I will add game_number to both datasets

## Load Datasets

In [ ]:
# Load the datasets
print("Loading all_seasons_data.csv...")
all_seasons_df = pd.read_csv('all_seasons_data.csv')

print("Loading defensive_stats_raw.csv...")
defensive_stats_df = pd.read_csv('defensive_stats_raw.csv')

print(f"\n✓ All seasons data shape: {all_seasons_df.shape}")
print(f"✓ Defensive stats shape: {defensive_stats_df.shape}")

print(f"\nSeasons in all_seasons_data: {sorted(all_seasons_df['season'].unique())}")
print(f"Seasons in defensive_stats: {sorted(defensive_stats_df['season'].unique())}")

## Step 1: Build Team-Game Mapping from fixtures.csv

For each season (2018-19 to 2024-25):
1. Load fixtures.csv and teams.csv
2. Create team_id → team_name mapping
3. Convert each fixture to 2 team-game records (home + away)
4. Sort chronologically and assign game_number 1-38

In [ ]:
def build_game_number_mapping():
    """
    Build a comprehensive mapping of (season, fixture_id) → game_number
    using fixtures.csv as the source of truth.
    
    Only processes seasons that have BOTH fixtures.csv AND teams.csv to ensure
    accurate team ID to name mapping.
    
    Returns:
        - fixture_to_game_number: DataFrame with (season, fixture_id, team_name, game_number)
    """
    
    # Season folders to process (only those with teams.csv)
    season_folders = ['2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25', '2025-26']
    
    all_mappings = []
    
    for season in season_folders:
        fixtures_path = f'data/{season}/fixtures.csv'
        teams_path = f'data/{season}/teams.csv'
        
        try:
            # Load fixtures and teams
            fixtures_df = pd.read_csv(fixtures_path)
            teams_df = pd.read_csv(teams_path)
            
            # Create team_id → team_name mapping
            team_id_to_name = dict(zip(teams_df['id'], teams_df['name']))
            
            # Convert kickoff_time to datetime
            fixtures_df['kickoff_time'] = pd.to_datetime(fixtures_df['kickoff_time'])
            
            # Keep only finished matches
            finished_fixtures = fixtures_df[fixtures_df['finished'] == True].copy()
            
            if len(finished_fixtures) == 0:
                print(f"{season}: No finished matches found, skipping")
                continue
            
            # Create team-game records (2 per fixture: home + away)
            team_games = []
            
            for _, fixture in finished_fixtures.iterrows():
                fixture_id = fixture['id']
                kickoff_time = fixture['kickoff_time']
                gw = fixture['event']
                
                # Home team record
                team_games.append({
                    'season': season,
                    'fixture_id': fixture_id,
                    'team_id': fixture['team_h'],
                    'team_name': team_id_to_name.get(fixture['team_h'], 'Unknown'),
                    'kickoff_time': kickoff_time,
                    'gw': gw,
                    'is_home': True
                })
                
                # Away team record
                team_games.append({
                    'season': season,
                    'fixture_id': fixture_id,
                    'team_id': fixture['team_a'],
                    'team_name': team_id_to_name.get(fixture['team_a'], 'Unknown'),
                    'kickoff_time': kickoff_time,
                    'gw': gw,
                    'is_home': False
                })
            
            season_df = pd.DataFrame(team_games)
            
            # Sort by team and kickoff_time (chronological order)
            season_df = season_df.sort_values(['team_name', 'kickoff_time'])
            
            # Assign game_number per team (1, 2, 3, ..., up to 38)
            season_df['game_number'] = season_df.groupby('team_name').cumcount() + 1
            
            all_mappings.append(season_df)
            
            print(f"{season}: {len(finished_fixtures)} fixtures → {len(season_df)} team-game records")
            print(f"         Teams: {season_df['team_name'].nunique()}, Max game_number: {season_df['game_number'].max()}")
            
        except FileNotFoundError as e:
            print(f"{season}: Required file not found, skipping ({e})")
        except Exception as e:
            print(f"{season}: Error - {e}")
    
    # Combine all seasons
    if all_mappings:
        full_mapping = pd.concat(all_mappings, ignore_index=True)
        print(f"\n✅ Total mapping records: {len(full_mapping):,}")
        print(f"Note: Seasons 2016-17, 2017-18, 2018-19 excluded (missing teams.csv)")
        return full_mapping
    else:
        print("❌ No mappings created!")
        return None

# Build the mapping
print("="*80)
print("STEP 1: Building game_number mapping from fixtures.csv")
print("="*80)
game_number_mapping = build_game_number_mapping()

## Step 2: Join game_number to all_seasons_data.csv

Join using the fixture ID (100% match rate for seasons with fixtures.csv)

In [ ]:
def add_game_number_to_all_seasons(df, mapping):
    """
    Add game_number to all_seasons_data using fixture ID matching.
    
    Args:
        df: all_seasons_data DataFrame
        mapping: game_number_mapping DataFrame from Step 1
        
    Returns:
        DataFrame with game_number column added
    """
    df = df.copy()
    
    # Drop existing game_number if present
    if 'game_number' in df.columns:
        df = df.drop('game_number', axis=1)
        print("Dropped existing game_number column")
    
    # Create slim mapping for joining: (season, fixture_id, team_name) → game_number
    # We need team_name because each fixture has 2 teams
    slim_mapping = mapping[['season', 'fixture_id', 'team_name', 'game_number']].copy()
    slim_mapping = slim_mapping.rename(columns={'fixture_id': 'fixture', 'team_name': 'team'})
    
    print(f"Original records: {len(df):,}")
    print(f"Mapping records: {len(slim_mapping):,}")
    
    # Merge on season, fixture, and team
    df = df.merge(
        slim_mapping,
        on=['season', 'fixture', 'team'],
        how='left'
    )
    
    # Convert to nullable integer
    df['game_number'] = df['game_number'].astype('Int64')
    
    # Report results
    matched = df['game_number'].notna().sum()
    unmatched = df['game_number'].isna().sum()
    
    print(f"\n✅ Matched records: {matched:,} ({matched/len(df)*100:.1f}%)")
    print(f"❌ Unmatched records: {unmatched:,} ({unmatched/len(df)*100:.1f}%)")
    
    # Check which seasons have unmatched records
    if unmatched > 0:
        unmatched_by_season = df[df['game_number'].isna()].groupby('season').size()
        print(f"\nUnmatched by season:")
        print(unmatched_by_season)
    
    return df

# Apply to all_seasons_data
print("="*80)
print("STEP 2: Adding game_number to all_seasons_data.csv")
print("="*80)
all_seasons_updated = add_game_number_to_all_seasons(all_seasons_df, game_number_mapping)

## Step 3: Add game_number to defensive_stats_raw.csv

Since defensive_stats doesn't have fixture IDs, we'll use date-based matching:
1. Parse the 'game' column to extract date and teams
2. Match with fixtures using date + team combination

In [ ]:
def add_game_number_to_defensive_stats(df, mapping):
    """
    Add game_number to defensive_stats using date-based matching.
    
    The 'game' column format: "YYYY-MM-DD Team1-Team2"
    The 'season' column format: "1920" (for 2019-20)
    
    Args:
        df: defensive_stats DataFrame
        mapping: game_number_mapping DataFrame
        
    Returns:
        DataFrame with game_number column added
    """
    df = df.copy()
    
    # Drop existing game_number if present
    if 'game_number' in df.columns:
        df = df.drop('game_number', axis=1)
        print("Dropped existing game_number column")
    
    # Team name mapping: defensive_stats name → mapping name (FPL short name)
    team_name_mapping = {
        'Brighton & Hove Albion': 'Brighton',
        'Ipswich Town': 'Ipswich',
        'Leeds United': 'Leeds',
        'Leicester City': 'Leicester',
        'Luton Town': 'Luton',
        'Manchester City': 'Man City',
        'Manchester United': 'Man Utd',
        'Newcastle United': 'Newcastle',
        'Norwich City': 'Norwich',
        'Nottingham Forest': "Nott'm Forest",
        'Sheffield United': 'Sheffield Utd',
        'Tottenham Hotspur': 'Spurs',
        'West Bromwich Albion': 'West Brom',
        'West Ham United': 'West Ham',
        'Wolverhampton Wanderers': 'Wolves',
    }
    
    # Map team names to match FPL naming
    df['team_mapped'] = df['team'].map(team_name_mapping).fillna(df['team'])
    
    # Extract date from 'game' column (format: "YYYY-MM-DD Team1-Team2")
    df['game_date'] = pd.to_datetime(df['game'].str.extract(r'^(\d{4}-\d{2}-\d{2})')[0], errors='coerce')
    
    # Convert defensive stats season format (1920) to standard format (2019-20)
    def convert_season(s):
        if pd.isna(s):
            return None
        s = str(s).replace('.0', '')
        if len(s) == 4:  # e.g., "1920"
            return f"20{s[:2]}-{s[2:]}"
        return s
    
    df['season_standard'] = df['season'].apply(convert_season)
    
    # Create date-based mapping from fixtures
    mapping_for_date = mapping.copy()
    mapping_for_date['game_date'] = mapping_for_date['kickoff_time'].dt.date
    mapping_for_date['game_date'] = pd.to_datetime(mapping_for_date['game_date'])
    
    # Create slim mapping: (season, team_name, game_date) → game_number
    date_mapping = mapping_for_date[['season', 'team_name', 'game_date', 'game_number']].copy()
    date_mapping = date_mapping.rename(columns={'team_name': 'team_mapped', 'season': 'season_standard'})
    
    # Remove duplicates (same team can't have 2 games on same day)
    date_mapping = date_mapping.drop_duplicates(subset=['season_standard', 'team_mapped', 'game_date'])
    
    print(f"Original records: {len(df):,}")
    print(f"Date mapping records: {len(date_mapping):,}")
    
    # Merge
    df = df.merge(
        date_mapping,
        on=['season_standard', 'team_mapped', 'game_date'],
        how='left'
    )
    
    # Convert to nullable integer
    df['game_number'] = df['game_number'].astype('Int64')
    
    # Clean up temporary columns
    df = df.drop(['game_date', 'season_standard', 'team_mapped'], axis=1)
    
    # Report results
    matched = df['game_number'].notna().sum()
    unmatched = df['game_number'].isna().sum()
    
    print(f"\n✅ Matched records: {matched:,} ({matched/len(df)*100:.1f}%)")
    print(f"❌ Unmatched records: {unmatched:,} ({unmatched/len(df)*100:.1f}%)")
    
    # Check which seasons have unmatched records
    if unmatched > 0:
        unmatched_by_season = df[df['game_number'].isna()].groupby('season').size()
        print(f"\nUnmatched by season:")
        print(unmatched_by_season)
    
    return df

# Apply to defensive_stats
print("="*80)
print("STEP 3: Adding game_number to defensive_stats_raw.csv")
print("="*80)
defensive_stats_updated = add_game_number_to_defensive_stats(defensive_stats_df, game_number_mapping)

## Step 5: Save Updated Datasets

In [ ]:
# Save the updated datasets
print("Saving updated datasets...")
print("="*80)

# Save all_seasons_data with game_number
all_seasons_updated.to_csv('all_seasons_data.csv', index=False)
print(f"✓ Saved: all_seasons_data.csv")
print(f"  - {len(all_seasons_updated):,} records")
print(f"  - game_number range: 1-{all_seasons_updated['game_number'].max()}")

# Save defensive_stats with game_number
defensive_stats_updated.to_csv('defensive_stats_raw.csv', index=False)
print(f"\n✓ Saved: defensive_stats_raw.csv")
print(f"  - {len(defensive_stats_updated):,} records")
print(f"  - game_number range: 1-{defensive_stats_updated['game_number'].max()}")

print("\n" + "="*80)
print("ALL DONE!")
print("="*80)

# now merging the main table with the defensive stats table

### read both tables

In [ ]:
# Load the raw defensive data
print("Loading defensive stats raw data...")
df_def = pd.read_csv('defensive_stats_raw.csv', low_memory=False)
print(f"Original defensive data shape: {df_def.shape}")

# Remove the header row (row 0 contains column descriptions)
df_def = df_def[df_def['season'].notna() & (df_def['season'] != '')]
df_def = df_def.reset_index(drop=True)
print(f"After removing header row: {df_def.shape}")

In [ ]:
# 1. STANDARDIZE SEASON FORMAT (1920 -> 2019-20)
def convert_season_format(season_code):
    """Convert season from '1920' to '2019-20' format"""
    if pd.isna(season_code) or season_code == '':
        return None
    try:
        season_str = str(int(float(season_code)))
        if len(season_str) == 4:
            year1 = int('20' + season_str[:2])
            year2 = season_str[2:]
            return f"{year1}-{year2}"
        return None
    except:
        return None

df_def['season'] = df_def['season'].apply(convert_season_format)
print(f"\nSeasons after conversion:")
print(df_def['season'].value_counts().sort_index())

In [ ]:
# 2. COMBINE TACKLE COLUMNS (Def 3rd, Mid 3rd, Att 3rd -> Total Tackles)
print("="*60)
print("COMBINING TACKLE COLUMNS")
print("="*60)

# The columns are named 'Tackles.2' (Def 3rd), 'Tackles.3' (Mid 3rd), 'Tackles.4' (Att 3rd)
# Convert to numeric and combine
tackle_cols = ['Tackles.2', 'Tackles.3', 'Tackles.4']
for col in tackle_cols:
    df_def[col] = pd.to_numeric(df_def[col], errors='coerce')

# Create combined tackles column (sum of all three thirds)
df_def['tackles_total'] = df_def[tackle_cols].sum(axis=1)

print(f"✓ Created 'tackles_total' by combining tackles from all thirds")
print(f"  Sample values: {df_def['tackles_total'].head(10).tolist()}")

In [ ]:
# 2. COMBINE TACKLE COLUMNS (Def 3rd, Mid 3rd, Att 3rd -> Total Tackles)
# The columns are named 'Tackles.2', 'Tackles.3', 'Tackles.4' representing Def 3rd, Mid 3rd, Att 3rd
print("="*60)
print("COMBINING TACKLE COLUMNS")
print("="*60)

# Convert tackle columns to numeric
tackle_cols = ['Tackles.2', 'Tackles.3', 'Tackles.4']  # Def 3rd, Mid 3rd, Att 3rd
for col in tackle_cols:
    df_def[col] = pd.to_numeric(df_def[col], errors='coerce')

# Create combined tackles column (sum of all three thirds)
df_def['tackles_total'] = df_def[tackle_cols].sum(axis=1)

print(f"Created 'tackles_total' column by combining:")
print(f"  - Tackles.2 (Def 3rd)")
print(f"  - Tackles.3 (Mid 3rd)")
print(f"  - Tackles.4 (Att 3rd)")
print(f"\nSample: {df_def['tackles_total'].describe()}")

In [ ]:
# 3. SELECT AND RENAME DEFENSIVE COLUMNS
print("="*60)
print("SELECTING DEFENSIVE COLUMNS")
print("="*60)

# Map raw columns to FPL-style naming
column_mapping = {
    'season': 'season',
    'player': 'name',
    'team': 'team',
    'pos': 'position',
    'min': 'minutes',
    'Tackles': 'tackles',  # Total tackles (Tkl)
    'Tackles.1': 'tackles_won',  # TklW
    'tackles_total': 'tackles_total',  # Combined Def+Mid+Att third
    'Challenges': 'challenges',  # Total challenges
    'Challenges.1': 'challenges_attempted',  # Att
    'Challenges.2': 'challenges_success_rate',  # Tkl%
    'Challenges.3': 'challenges_lost',  # Lost
    'Blocks': 'blocks',  # Total blocks
    'Blocks.1': 'blocks_shots',  # Sh
    'Blocks.2': 'blocks_passes',  # Pass
    'Int': 'interceptions',  # Interceptions
    'Tkl+Int': 'tackles_interceptions',  # Tkl+Int
    'Clr': 'clearances',  # Clearances
    'Err': 'errors',  # Errors leading to shots
    'match_id': 'match_id',
    'game': 'game'
}

# Select only the columns we need for defensive stats
defensive_columns = list(column_mapping.keys())
df_def_selected = df_def[defensive_columns].copy()

# Rename columns to match FPL naming
df_def_selected = df_def_selected.rename(columns=column_mapping)

print(f"Selected columns: {df_def_selected.columns.tolist()}")

In [ ]:
# 4. CONVERT DATA TYPES AND FILL MISSING VALUES
print("="*60)
print("CONVERTING DATA TYPES")
print("="*60)

# Numeric columns
numeric_cols = [
    'minutes', 'tackles', 'tackles_won', 'tackles_total',
    'challenges', 'challenges_attempted', 'challenges_success_rate', 
    'challenges_lost', 'blocks', 'blocks_shots', 'blocks_passes',
    'interceptions', 'tackles_interceptions', 'clearances', 'errors'
]

for col in numeric_cols:
    df_def_selected[col] = pd.to_numeric(df_def_selected[col], errors='coerce')

# Fill NaN values with 0 for defensive stats (no stat = 0)
df_def_selected[numeric_cols] = df_def_selected[numeric_cols].fillna(0)

print("✓ Converted numeric columns and filled NaN with 0")

# Assign gameweek

In [ ]:
# 5. EXTRACT GAMEWEEK FROM MATCH DATA
print("="*60)
print("EXTRACTING GAMEWEEK INFORMATION")
print("="*60)

# Parse the 'game' column to extract date (Format: "2019-08-09 Liverpool-Norwich City")
def extract_date(game_str):
    """Extract date from game string"""
    if pd.isna(game_str) or game_str == '':
        return None
    try:
        parts = str(game_str).split(' ')
        if len(parts) > 0:
            return parts[0]  # Return the date part
    except:
        pass
    return None

df_def_selected['game_date'] = df_def_selected['game'].apply(extract_date)
df_def_selected['game_date'] = pd.to_datetime(df_def_selected['game_date'], errors='coerce')

# Assign gameweek based on season and date
def assign_gameweek(row):
    """Assign gameweek based on season and date"""
    if pd.isna(row['game_date']) or pd.isna(row['season']):
        return None
    
    season = row['season']
    game_date = row['game_date']
    
    try:
        year = int(season.split('-')[0])
    except:
        return None
    
    # Season typically starts in early August
    season_start = pd.Timestamp(f'{year}-08-01')
    
    # Calculate weeks from season start
    weeks_diff = (game_date - season_start).days // 7
    
    # Gameweek is approximately weeks + 1, capped at 38
    gw = min(max(weeks_diff + 1, 1), 38)
    
    return int(gw)

df_def_selected['GW'] = df_def_selected.apply(assign_gameweek, axis=1)

print(f"✓ Assigned gameweeks. GW range: {df_def_selected['GW'].min()} to {df_def_selected['GW'].max()}")

## save the cleaned data into csv

In [ ]:
df_def_selected.to_csv("defensive_stats_cleaned.csv", index=False)

# Now the merging part:

### load both datasets

In [ ]:
# loading both datasets
df_def = pd.read_csv("defensive_stats_cleaned.csv")
df_main = pd.read_csv("all_seasons_data.csv")

### cleaning the name field 

In [ ]:

# ==========================================
# STEP 1: FORCE-CLEAN NAMES
# ==========================================

def clean_main_name_format(name):
    if not isinstance(name, str):
        return str(name)
    
    # 1. Fix Mojibake (Encoding Errors)
    char_map = {
        'Ã©': 'é', 'Ãº': 'ú', 'Ã¡': 'á', 'Ã³': 'ó', 'Ã¨': 'è', 'Ã±': 'ñ',
        'Ã\xad': 'í', 'Ã§': 'ç', 'Ã¢': 'â', 'Ã¼': 'ü', 'Ã¶': 'ö', 'Ã\x9f': 'ß',
        'Ã¸': 'ø', 'Ã«': 'ë', 'Ã': 'à' ,'à£': 'ã', 'à©': 'é'
    }
    for bad, good in char_map.items():
        name = name.replace(bad, good)

    # 2. Remove trailing IDs (e.g., '_376', '_12', '_4')
    # Regex: Underscore followed by 1 or more digits at the END of the string
    name = re.sub(r'_\d+$', '', name)
    
    # 3. Replace remaining underscores with spaces
    name = name.replace('_', ' ')
    
    # 4. Standardize (lower, strip)
    return name.lower().strip()

# Create/Overwrite the 'join_name' column
df_main['join_name'] = df_main['name'].apply(clean_main_name_format)


### filter only needed seasons

In [ ]:
# first filter only the seasons that we need from defensive data
seasons_needed = df_def['season'].unique()
df_main = df_main[df_main['season'].isin(seasons_needed)]
# print the target seasons
print("Seasons needed for merging:", seasons_needed)

### adding clearance_block_interception col

In [ ]:
#clearances_blocks_interceptions, recoveries, defensive_contribution, tackles are the cols needed to be added to the defensive df

def add_clearances_blocks_interceptions(df):
    df['clearances_blocks_interceptions'] = df['clearances'] + df['blocks'] + df['interceptions']
    return df
df_def = add_clearances_blocks_interceptions(df_def)
# print sample data to verify
print(df_def[['clearances', 'blocks', 'interceptions', 'clearances_blocks_interceptions']].head())

## standarizing the keys for perfect matching

In [ ]:
df_def['join_name'] = df_def['name'].astype(str).str.lower().str.strip()

# 2. Standardize Seasons (Ensure they match "2023-24" format in both)
df_main['join_season'] = df_main['season'].astype(str).str.strip()
df_def['join_season'] = df_def['season'].astype(str).str.strip()

manual_nickname_map = {
    'jorge luiz frello filho': 'jorginho',
    'jonathan castro otto': 'jonny',
    'bruno guimaraes rodriguez moura': 'bruno guimaraes',
    'gabriel teodoro martinelli silva': 'gabriel martinelli',
    'emerson leie de souza junior': 'emerson royal',
    'david raya martin': 'david raya',
    'jose sa': 'jose sa',
    'joao palhinha goncalves alves': 'joao palhinha',
    'thiago alcantara do nascimento': 'thiago',
    'matheus luiz nunes': 'matheus nunes',
    'antony matheus dos santos': 'antony',
    'richarlison de andrade': 'richarlison',
    'bernardo mota veiga de carvalho e silva': 'bernardo silva',
    'ederson santana de moraes': 'ederson',
    'joão filipe iria santos moutinho': 'joão moutinho',
    'fabio henrique tavares': 'fabinho',
    'frederico rodrigues de paula santos': 'fred',
    'lucas tolentino coelho de lima': 'lucas paquetá', # Check accent
    'lucas tolentino coelho de lima': 'lucas paqueta', # Try both if unsure
    'benjamin white': 'ben white',
    'gabriel dos santos magalhães': 'gabriel magalhães', # Now that encoding is fixed, map to short
    'bruno guimarães rodriguez moura': 'bruno guimarães',
    'joão palhinha gonçalves': 'joão palhinha',
    'tomas soucek': 'tomáš souček', # Add accents if defensive has them
    'casemiro': 'casemiro', # If Main has long name, map it. Likely 'carlos henrique casimiro'
    'carlos henrique casimiro': 'casemiro',
    'norberto bercique gomes betuncal': 'beto',
    'jose angel esmoris tasende': 'angelino',
    'juan camilo hernandez suarez': 'cucho',
    'daniel ceballos fernandez': 'dani ceballos',
    'anssumane fati vieira': 'ansu fati',
    'anssumane fati': 'ansu fati',
    'alexandre moreno lopera': 'alex moreno',
    'łukasz fabianski': 'lukasz fabianski', # Fix the Polish 'ł' manually
    'lukasz fabianski': 'lukasz fabianski',  # Safety net
    'ahmed el-sayed hegazy': 'ahmed hegazi',
    'a\x81lex moreno lopera': 'alex moreno',   # Found in your list
    'ivan peria¡ia\x87': 'ivan perisic',       # Found in your list
    'muhamed bea¡ia\x87': 'muhamed besic',     # Found in your list
    'a\x81lex moreno lopera': 'alex moreno',
    'edson a\x81lvarez velazquez': 'edson alvarez',
    
    # The Nicknames & Legal Names
    'abdul fatawu': 'abdul fatawu issahaku',
    'borja gonzalez tomas': 'borja baston',
    'fabio ferreira vieira': 'fabio vieira',
    'fernando luiz rosa': 'fernandinho',
    'giovanni reyna': 'gio reyna',
    'hamed traore': 'hamed junior traore',
    'ian carlo poveda-ocampo': 'ian poveda',
    'jhon duran': 'jader duran',
    'julian araujo zuniga': 'julian araujo',
    'thakgalo leshabela': 'khanya leshabela',
    'francisco casilla cortes': 'kiko casilla',
    'francisco femenia far': 'kiko femenia',
    'marcus oliveira alencar': 'marquinhos',
    'oluwasemilogo adesewo ibidapo ajayi': 'semi ajayi',
    'tariqe fosu-henry': 'tariqe fosu',
    'vini de souza costa': 'vinicius souza',
    'vitor ferreira': 'vitinha',
    'jose reina': 'pepe reina',
    'djordje petrovic': 'đorđe petrovic',  # Matching the defensive spelling
    'jose a\x81ngel esmoris tasende': 'angelino', # Found hidden in candidate list
}

df_main['join_name'] = df_main['join_name'].replace(manual_nickname_map)


# ==========================================
# STEP 3: AUTOMATED SMART MATCHING
# ==========================================
print("--- STARTING SMART MATCHING ---")

# 1. PREPARE LISTS
# We only care about names that are currently missing in the Main DF
# (i.e., names in Main that don't yet match a name in Defensive)
valid_def_names = set(df_def['join_name'].unique())
main_unique = df_main['join_name'].unique()
missing_names = [n for n in main_unique if n not in valid_def_names]

print(f"Attempting to resolve {len(missing_names)} missing names...")

name_mapping = {}

# 2. LOGIC A: SUBSTRING MATCH (The "Gabriel Jesus" Fix)
# We check if a Defensive Name (Short) is fully inside a Main Name (Long)
# e.g. "gabriel jesus" is inside "gabriel fernando de jesus"
for m_name in missing_names:
    m_tokens = set(m_name.split())
    candidates = []
    
    for d_name in valid_def_names:
        d_tokens = set(d_name.split())
        # Check if ALL words in the short name appear in the long name
        if d_tokens.issubset(m_tokens):
            candidates.append(d_name)
    
    if candidates:
        # If multiple matches, pick the longest one (Most specific)
        # Prevents "Gabriel" matching "Gabriel Jesus" incorrectly
        best_match = max(candidates, key=len)
        name_mapping[m_name] = best_match

# 3. LOGIC B: FUZZY MATCH (The "Fabian Schär" Fix)
# For names that didn't match via substring (likely due to spelling/encoding diffs)
# We only check names that Logic A didn't solve
remaining_missing = [n for n in missing_names if n not in name_mapping]
def_name_list = list(valid_def_names)

for m_name in remaining_missing:
    # Cutoff 0.8 is strict to avoid bad matches (we prefer missing data over wrong data)
    matches = difflib.get_close_matches(m_name, def_name_list, n=1, cutoff=0.8)
    if matches:
        name_mapping[m_name] = matches[0]

# 4. APPLY THE UPDATES (To the JOIN KEY only)
print(f"Found {len(name_mapping)} new automatic matches.")
print("Updating 'join_name' column (Original names are safe)...")
df_main['join_name'] = df_main['join_name'].replace(name_mapping)


# ==========================================
# 1. DEFINE ACCENT REMOVER
# ==========================================
def remove_accents(input_str):
    if not isinstance(input_str, str):
        return str(input_str)
    # Normalize unicode characters to decompose them (e.g., 'á' becomes 'a' + '´')
    nfkd_form = unicodedata.normalize('NFKD', input_str)
    # Filter out non-spacing mark characters (the accents)
    return "".join([c for c in nfkd_form if not unicodedata.combining(c)])

print("--- STRIPPING ACCENTS FROM BOTH DATASETS ---")

# Apply to Main
df_main['join_name'] = df_main['join_name'].apply(remove_accents)

# Apply to Defensive
df_def['join_name'] = df_def['join_name'].apply(remove_accents)



### coverage report:

In [ ]:
print("\n=== UNIQUE NAME COVERAGE REPORT ===")

# Get unique names
main_names_set = set(df_main['join_name'].unique())
def_names_set = set(df_def['join_name'].unique())

# Calculate intersection
matched_names = main_names_set.intersection(def_names_set)
missing_def_names = def_names_set - main_names_set

# Metrics
total_def_names = len(def_names_set)
matched_count = len(matched_names)
coverage_pct = (matched_count / total_def_names) * 100

print(f"Unique Defensive Names:   {total_def_names}")
print(f"Found in Main DataFrame:  {matched_count}")
print(f"Defensive Name Coverage:  {coverage_pct:.2f}%")

# now the real merging:

In [ ]:
# ==========================================
# 2. THE MERGE & SMART VALIDATION
# ==========================================
# IMPORTANT: We now use game_number instead of GW for merging
# This handles postponed matches correctly by matching on chronological game order
print("--- MERGE DIAGNOSTICS (Using game_number) ---")

# 1. PREPARE DEFENSIVE DATA
cols_cbi = ['clearances', 'blocks', 'interceptions']
df_def[cols_cbi] = df_def[cols_cbi].fillna(0)

if 'clearances_blocks_interceptions' not in df_def.columns:
    df_def['clearances_blocks_interceptions'] = (
        df_def['clearances'] + df_def['blocks'] + df_def['interceptions']
    )

# Select Merge Subset - NOW USING game_number INSTEAD OF GW
def_subset = df_def[[
    'join_name', 'game_number', 'join_season', 
    'tackles', 'clearances_blocks_interceptions'
]].rename(columns={
    'tackles': 'tackles_new', 
    'clearances_blocks_interceptions': 'cbi_new'
})

# ---------------------------------------------------------
# SMART METRIC: "Can we match it?"
# ---------------------------------------------------------
# Using game_number for matching ensures correct alignment even with postponed matches
main_keys = set(zip(df_main['join_name'], df_main['game_number'], df_main['join_season']))
def_keys = set(zip(def_subset['join_name'], def_subset['game_number'], def_subset['join_season']))

# The Intersection: These are the rows that SHOULD merge successfully
possible_matches = main_keys.intersection(def_keys)
print(f"Total Rows in Main: {len(df_main)}")
print(f"Rows with available Defensive Data: {len(possible_matches)}")

# 2. PERFORM LEFT MERGE - NOW ON game_number
merged_df = pd.merge(
    df_main, 
    def_subset, 
    on=['join_name', 'game_number', 'join_season'], 
    how='left'
)

# ---------------------------------------------------------
# REAL VALIDATION: DID THE MERGE WORK?
# ---------------------------------------------------------
merged_df['key_tuple'] = list(zip(merged_df['join_name'], merged_df['game_number'], merged_df['join_season']))
should_have_data = merged_df[merged_df['key_tuple'].isin(possible_matches)]

# Check if they are actually filled
successful_merges = should_have_data['tackles_new'].notna().sum()
technical_success_rate = (successful_merges / len(should_have_data)) * 100 if len(should_have_data) > 0 else 0

print(f"\nTechnical Merge Success Rate: {technical_success_rate:.2f}%")
print("(This should be 100%. It means every row that existed in the source was successfully merged.)")

# ==========================================
# 3. UPDATE STATS & FINAL REPORT
# ==========================================
# Update Tackles
merged_df['tackles'] = np.where(
    merged_df['tackles_new'].notna(), 
    merged_df['tackles_new'], 
    np.where(merged_df['minutes'] == 0, 0, merged_df['tackles'].fillna(0))
)

# Update CBI
old_cbi = merged_df['clearances_blocks_interceptions'] if 'clearances_blocks_interceptions' in merged_df.columns else 0
merged_df['clearances_blocks_interceptions'] = np.where(
    merged_df['cbi_new'].notna(), 
    merged_df['cbi_new'], 
    np.where(merged_df['minutes'] == 0, 0, old_cbi)
)

# Clean up temps - keep game_number as it's useful for feature engineering
merged_df.drop(columns=['tackles_new', 'cbi_new', 'join_name', 'join_season', 'is_matched', 'key_tuple'], inplace=True, errors='ignore')

print("\n✓ Merge complete using game_number for correct chronological alignment")

# here validate that after merging we made the results in all seasons data

In [ ]:
# ==========================================
# STEP 4: COMBINE MERGED DATA BACK INTO FULL DATASET
# ==========================================
# Problem: We filtered df_main to only seasons with defensive data (2019-20 to 2024-25)
# Solution: Reload the full dataset and combine:
#   - Seasons 2016-17 to 2018-19: Already have real defensive stats
#   - Seasons 2019-20 to 2024-25: Use merged_df with newly merged defensive stats
#   - Season 2025-26: Keep as-is (if not in defensive data)

print("="*80)
print("STEP 4: Combining merged data back into full dataset")
print("="*80)

# 1. Reload the full all_seasons_data (before we filtered it)
df_full = pd.read_csv("all_seasons_data.csv")
print(f"Full dataset shape: {df_full.shape}")
print(f"Seasons in full dataset: {sorted(df_full['season'].unique())}")

# 2. Get the seasons that were merged with defensive data
merged_seasons = merged_df['season'].unique().tolist()
print(f"\nSeasons that were merged with defensive stats: {merged_seasons}")

# 3. Get the seasons that were NOT merged (already have defensive data or no data available)
seasons_not_merged = [s for s in df_full['season'].unique() if s not in merged_seasons]
print(f"Seasons NOT merged (already have defensive data): {seasons_not_merged}")

# 4. Extract the non-merged seasons from the full dataset
df_not_merged = df_full[df_full['season'].isin(seasons_not_merged)].copy()
print(f"Records from non-merged seasons: {len(df_not_merged):,}")

# 5. Ensure both DataFrames have the same columns
# Add game_number to non-merged seasons if missing (set to None for older seasons)
if 'game_number' not in df_not_merged.columns:
    df_not_merged['game_number'] = None
    
# Align columns - get the union of both column sets
all_columns = list(set(merged_df.columns) | set(df_not_merged.columns))
for col in all_columns:
    if col not in merged_df.columns:
        merged_df[col] = None
    if col not in df_not_merged.columns:
        df_not_merged[col] = None

# 6. Combine merged seasons with non-merged seasons
all_seasons_final = pd.concat([df_not_merged, merged_df], ignore_index=True)

# 7. Sort by season and element for consistency
all_seasons_final = all_seasons_final.sort_values(['season', 'element', 'GW']).reset_index(drop=True)

print(f"\n✅ Final combined dataset shape: {all_seasons_final.shape}")
print(f"Seasons in final dataset: {sorted(all_seasons_final['season'].unique())}")

# 8. Verify record counts by season
print("\nRecords per season:")
print(all_seasons_final.groupby('season').size().sort_index())

## Step 5: Recalculate defensive_contribution and modify points for merged seasons

In [ ]:
# ==========================================
# STEP 5: RECALCULATE DEFENSIVE CONTRIBUTION & MODIFY POINTS
# ==========================================
# Now that we have the real defensive stats for 2019-20 to 2024-25,
# we need to recalculate defensive_contribution and apply the point modifications

print("="*80)
print("STEP 5: Recalculating defensive contribution for merged seasons")
print("="*80)

# 1. Define the defensive contribution calculation function
def calculate_defensive_contribution_row(row):
    """Calculate defensive contribution based on position"""
    position = row['position']
    cbi = row.get('clearances_blocks_interceptions', 0) or 0
    tackles = row.get('tackles', 0) or 0
    recoveries = row.get('recoveries', 0) or 0
    
    if position == 'DEF':
        return cbi + tackles
    elif position in ['MID', 'FWD']:
        return cbi + tackles + recoveries
    else:  # GK or unknown
        return 0

# 2. Recalculate defensive_contribution for seasons that were merged
# (2019-20 to 2024-25 now have real defensive stats)
merged_season_mask = all_seasons_final['season'].isin(merged_seasons)

print(f"Recalculating defensive_contribution for {merged_season_mask.sum():,} records...")

all_seasons_final.loc[merged_season_mask, 'defensive_contribution'] = \
    all_seasons_final.loc[merged_season_mask].apply(calculate_defensive_contribution_row, axis=1)

# 3. Apply point modification to merged seasons (as per FPL rules)
# Add 2 bonus points when:
# - Defenders reach defensive_contribution >= 10
# - Midfielders/Forwards reach defensive_contribution >= 12

def calculate_modified_points(row):
    """Add 2 bonus points for defensive contribution threshold"""
    points = row['total_points']
    position = row['position']
    def_contrib = row.get('defensive_contribution', 0) or 0
    
    if position == 'DEF' and def_contrib >= 10:
        points += 2
    elif position in ['MID', 'FWD'] and def_contrib >= 12:
        points += 2
    return points

# Only modify points for the merged seasons (2019-20 to 2024-25)
# Seasons 2016-17 to 2018-19 already had points modified earlier
print("Applying point modifications for defensive contributions...")

all_seasons_final.loc[merged_season_mask, 'total_points'] = \
    all_seasons_final.loc[merged_season_mask].apply(calculate_modified_points, axis=1)

print("✅ Defensive contribution recalculated and points modified")

# 4. Verify the results
print("\nSample of defensive stats after recalculation:")
sample_cols = ['name', 'season', 'position', 'tackles', 'clearances_blocks_interceptions', 
               'defensive_contribution', 'total_points']
available_cols = [c for c in sample_cols if c in all_seasons_final.columns]
print(all_seasons_final[all_seasons_final['season'] == '2023-24'][available_cols].head(10))

## Step 6: Save the final complete dataset

In [ ]:
# ==========================================
# STEP 6: SAVE THE FINAL COMPLETE DATASET
# ==========================================
print("="*80)
print("STEP 6: Saving final dataset")
print("="*80)

# Save the complete dataset with all seasons and merged defensive stats
output_path = 'all_seasons_data_final.csv'
all_seasons_final.to_csv(output_path, index=False, encoding='utf-8')

print(f"✅ Saved: {output_path}")
print(f"   Total records: {len(all_seasons_final):,}")
print(f"   Total columns: {len(all_seasons_final.columns)}")
print(f"   Seasons: {sorted(all_seasons_final['season'].unique())}")

# Summary statistics
print("\n" + "="*80)
print("FINAL DATASET SUMMARY")
print("="*80)
print(f"\nRecords by season:")
print(all_seasons_final.groupby('season').size().sort_index())

print(f"\nDefensive stats coverage (non-zero defensive_contribution):")
def_contrib_stats = all_seasons_final.groupby('season').apply(
    lambda x: (x['defensive_contribution'] > 0).sum() / len(x) * 100
)
print(def_contrib_stats.round(1).to_string())

print("\n" + "="*80)
print("DATA PREPARATION COMPLETE - READY FOR FEATURE ENGINEERING")
print("="*80)

# now we make the previous game and that stuff: